# Fine-tuning de um VLM com LoRA: Qwen2-VL-2B-Instruct no DocVQA

Este notebook mostra, passo a passo, como adaptar um **Vision-Language Model (VLM)** com **LoRA** (*Low-Rank Adaptation*) para responder perguntas sobre imagens de documentos (**DocVQA**), em uma GPU gratuita (T4 do Colab ou do Kaggle).

**Modelo:** `Qwen/Qwen2-VL-2B-Instruct` · **Dataset:** `HuggingFaceM4/DocumentVQA` (treino e validação com respostas públicas).

Experimento: o modelo é adaptado com LoRA em um subconjunto do split de treino (N_TRAIN exemplos) e avaliado em N_EVAL exemplos da validação, com o objetivo de entender o pipeline.

### Três cenários de avaliação
Em todos os cenários o modelo esta em 4 bits
1. **Base**: pergunta simples.
2. **Base + *prompt engineering***: modelo original, com uma instrução pedindo resposta curta.
3. **LoRA**: modelo adaptado, pergunta simples.

Assim é possível comparar o que muda ao alterar apenas o *prompt* com o que muda ao alterar os pesos.

### Antes de rodar
- **Colab:** *Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)*.
- **Kaggle:** ative **Internet** e escolha **GPU T4 ×2** ou **P100** em *Accelerator*. O notebook usa **uma** GPU (`device_map={"": 0}`).
- A T4 e a P100 **não têm bfloat16 nativo**, por isso o notebook usa **fp16**.

In [1]:
# 1) Instalação das dependências
!pip install -q transformers==5.21.0 accelerate==1.15.0 peft==0.21.2 bitsandbytes==0.50.2 datasets==5.1.0 rapidfuzz==3.14.6 "pillow>=10.0.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


## 2) Configuração

Todos os parâmetros do experimento ficam aqui, para que seja fácil repetir a execução e registrar o que foi usado.

In [2]:
# ===================== CONFIGURAÇÃO (edite aqui) =====================
MODEL_ID   = "Qwen/Qwen2-VL-2B-Instruct"
DATASET_ID = "HuggingFaceM4/DocumentVQA"

N_TRAIN    = 500       # nº de exemplos de treino
N_EVAL     = 100       # nº de exemplos da VALIDAÇÃO usados na avaliação
EPOCHS     = 3
MAX_STEPS  = -1        # >0 limita o nº de passos (útil para medir o tempo por passo)

USE_4BIT   = True      # True = QLoRA (4-bit); False = LoRA puro em fp16 (isola o efeito do LoRA)

# Resolução: cada região de 28x28 pixels vira ~1 token visual.
# Menos pixels = menos memória, mas texto pequeno em documentos fica ilegível.
MIN_PIXELS = 256       # em unidades de 28x28
MAX_PIXELS = 640       # reduza (ex.: 400) se estourar a memória
N_GPUS        = 2
BATCH_SIZE    = 4      # lote por passo (mantenha 1 por T4)
GRAD_ACC      = 4    # acumulação de gradiente -> lote efetivo = BATCH_SIZE * GRAD_ACC
LEARNING_RATE = 2e-4
LORA_R        = 64
LORA_ALPHA    = 128
SEED          = 42

# Instrução usada no cenário "base + prompt engineering" (pede resposta curta, como o DocVQA espera)
PROMPT_SUFFIX = "\nAnswer the question using a single word or phrase."

In [3]:
import json
CFG_KEYS = ["MODEL_ID", "DATASET_ID", "N_TRAIN", "EPOCHS", "MAX_STEPS",
            "USE_4BIT", "MIN_PIXELS", "MAX_PIXELS", "N_GPUS", "BATCH_SIZE", "GRAD_ACC",
            "LEARNING_RATE", "LORA_R", "LORA_ALPHA", "SEED"]
json.dump({k: globals()[k] for k in CFG_KEYS}, open("cfg.json", "w"))

In [4]:
# 3) Imports
import os
# Deve ser definido ANTES de importar o torch (reduz a fragmentação de memória da GPU)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import time
import importlib.metadata as md
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(SEED)

for pkg in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "datasets", "rapidfuzz"]:
    try:
        print(f"{pkg}=={md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}: não instalado")

torch==2.11.0+cu128
transformers==5.21.0
peft==0.21.2
bitsandbytes==0.50.2
accelerate==1.15.0
datasets==5.1.0
rapidfuzz==3.14.6


## 4) Código compartilhado (`common.py`)

O *processor*, a leitura do dataset, o *data collator* e a configuração de quantização são usados tanto neste notebook quanto no `train.py`. Para não duplicar código, ficam em um arquivo `common.py`, escrito pela célula abaixo e importado pelos dois.

In [5]:
%%writefile common.py
import torch
from datasets import load_dataset
from transformers import AutoProcessor, BitsAndBytesConfig


def make_processor(model_id, min_pixels, max_pixels):
    return AutoProcessor.from_pretrained(
        model_id,
        min_pixels=min_pixels * 28 * 28,
        max_pixels=max_pixels * 28 * 28,
    )


def make_bnb_config(use_4bit, dtype):
    if not use_4bit:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )


class ListDataset(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]


def take_examples(dataset_id, split, n, seed):
    # streaming: não baixa o split inteiro; as imagens ficam codificadas até serem usadas
    stream = load_dataset(dataset_id, split=split, streaming=True)
    stream = stream.shuffle(seed=seed, buffer_size=500).take(n)
    rows = [{"image": ex["image"], "question": ex["question"], "answers": ex["answers"]} for ex in stream]
    return ListDataset(rows)


def build_conversation(example, with_answer=True, suffix=""):
    conv = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": example["question"] + suffix},
        ],
    }]
    if with_answer:
        conv.append({
            "role": "assistant",
            "content": [{"type": "text", "text": str(example["answers"][0])}],
        })
    return conv


class VQACollator:
    def __init__(self, processor):
        self.processor = processor
        tok = processor.tokenizer
        self.im_start = tok.convert_tokens_to_ids("<|im_start|>")
        self.assistant_len = len(tok("assistant\n", add_special_tokens=False)["input_ids"])

    def __call__(self, batch):
        texts, images = [], []
        for example in batch:
            conv = build_conversation(example, with_answer=True)
            texts.append(self.processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=False))
            images.append(example["image"].convert("RGB"))

        inputs = self.processor(text=texts, images=images, padding=True, return_tensors="pt")

        input_ids = inputs["input_ids"]
        labels = input_ids.clone()
        labels[inputs["attention_mask"] == 0] = -100
        for i in range(labels.size(0)):
            starts = (input_ids[i] == self.im_start).nonzero().flatten()
            resp_start = starts[-1].item() + 1 + self.assistant_len
            labels[i, :resp_start] = -100
        inputs["labels"] = labels
        return inputs

Writing common.py


## 5) Processador e quantização

- **fp16** em vez de bf16, porque a T4 não tem bfloat16 nativo.
- **`USE_4BIT=True`** carrega o modelo quantizado em 4-bit (QLoRA).
- No `train.py`, **`device_map={"": local_rank}`** coloca um processo em cada GPU (importante no Kaggle com 2 T4), e o *gradient checkpointing* troca tempo de processamento por memória.

In [6]:
import importlib
importlib.invalidate_caches()   # garante que o common.py recém-escrito seja encontrado
from common import make_processor, make_bnb_config, take_examples, build_conversation, VQACollator

DTYPE = torch.float16   # T4 e P100 não têm bfloat16 nativo

processor = make_processor(MODEL_ID, MIN_PIXELS, MAX_PIXELS)
bnb_config = make_bnb_config(USE_4BIT, DTYPE)

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 6) Preparando o dataset (DocVQA)

Cada exemplo tem uma **imagem de documento**, uma **pergunta** e uma **lista de respostas aceitas** (`answers`). No treino usamos a primeira resposta; na avaliação (ANLS) vale a melhor entre todas.

- **Experimento:** lemos só `N_TRAIN` exemplos em *streaming*, sem baixar o dataset inteiro (cerca de 10 GB).

In [7]:
dataset = take_examples(DATASET_ID, "train", N_TRAIN, seed=SEED)
eval_dataset = take_examples(DATASET_ID, "validation", N_EVAL, seed=0)

ex = dataset[0]
print(f"Treino: {len(dataset)} exemplos | Avaliação: {len(eval_dataset)} exemplos")
print("Pergunta:", ex["question"])
print("Respostas aceitas:", ex["answers"])
print("Tamanho da imagem original:", ex["image"].size)

README.md:   0%|          | 0.00/806 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

... (4 outputs left out)

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Treino: 500 exemplos | Avaliação: 100 exemplos
Pergunta: Which town is Lazlo Toth a resident of?
Respostas aceitas: ['Thornton']
Tamanho da imagem original: (1670, 2164)


## 7) Data collator multimodal

Monta o *prompt* no formato de chat do Qwen2-VL (imagem + pergunta + resposta) e cria os *labels* **mascarando tudo que vem antes da resposta** (sistema, imagem e pergunta) e o *padding*. Assim, a perda é calculada apenas sobre a resposta do assistente.

O código do *collator* está em `common.py` (seção 4).

In [8]:
collator = VQACollator(processor)

In [9]:
# Verificação (rode uma vez): só a resposta deve ter label != -100
_b = collator([dataset[0]])
_ids, _lab = _b["input_ids"][0], _b["labels"][0]
img_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
print("Tokens na sequência:", _ids.numel(), "| tokens de imagem:", int((_ids == img_id).sum()))
print("Tokens supervisionados:", int((_lab != -100).sum()))
print("Texto supervisionado:", repr(processor.tokenizer.decode(_lab[_lab != -100])))
print("Resposta esperada   :", dataset[0]["answers"][0])

Tokens na sequência: 652 | tokens de imagem: 616
Tokens supervisionados: 4
Texto supervisionado: 'Thornton<|im_end|>\n'
Resposta esperada   : Thornton


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


In [10]:
%%writefile train.py
import os, json, math
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import (
    Qwen2VLForConditionalGeneration,
    TrainingArguments, Trainer, set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from common import make_processor, make_bnb_config, take_examples, VQACollator

globals().update(json.load(open("cfg.json")))   # parâmetros vindos do notebook
set_seed(SEED)

local_rank = int(os.environ.get("LOCAL_RANK", 0))
torch.cuda.set_device(local_rank)
DTYPE = torch.float16

processor = make_processor(MODEL_ID, MIN_PIXELS, MAX_PIXELS)
dataset = take_examples(DATASET_ID, "train", N_TRAIN, seed=SEED)
collator = VQACollator(processor)

# ---------------- modelo + LoRA ----------------
bnb_config = make_bnb_config(USE_4BIT, DTYPE)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=DTYPE,
    device_map={"": local_rank},
    attn_implementation="sdpa",
)
model.config.use_cache = False

if USE_4BIT:
    model = prepare_model_for_kbit_training(
        model, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
else:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules = (
    r".*layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
    r"|.*visual\.blocks\.\d+\.(attn\.qkv|attn\.proj|mlp\.fc1|mlp\.fc2)"
    r"|.*visual\.merger\.mlp\.(0|2)"
),
)
model = get_peft_model(model, lora_config)
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
model.print_trainable_parameters()

# ---------------- treino ----------------
steps_per_epoch = math.ceil(len(dataset) / (BATCH_SIZE * GRAD_ACC * N_GPUS))
total_steps = MAX_STEPS if MAX_STEPS > 0 else steps_per_epoch * EPOCHS
WARMUP_STEPS = max(1, int(0.03 * total_steps))

training_args = TrainingArguments(
    output_dir="./qwen2vl-lora-output",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    logging_steps=5,
    save_strategy="no",              # o adaptador é salvo ao final (treino curto: sem checkpoints)
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    dataloader_num_workers=0,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
)
trainer = Trainer(model=model, args=training_args, train_dataset=dataset, data_collator=collator)

torch.cuda.reset_peak_memory_stats()
train_result = trainer.train()

if trainer.is_world_process_zero():
    trainable, total = model.get_nb_trainable_parameters()
    cost = {
        "gpu": torch.cuda.get_device_name(local_rank),
        "n_gpus": N_GPUS,
        "usa_4bit": USE_4BIT,
        "max_pixels": MAX_PIXELS * 28 * 28,
        "exemplos_treino": len(dataset),
        "epocas": EPOCHS,
        "lote_efetivo": BATCH_SIZE * GRAD_ACC * N_GPUS,
        "tempo_treino_s": round(train_result.metrics["train_runtime"], 1),
        "pico_memoria_GiB": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "perda_final": round(train_result.metrics.get("train_loss", float("nan")), 4),
        "parametros_treinaveis": trainable,
        "parametros_totais": total,
    }
    print(cost)
    json.dump(cost, open("cost.json", "w"))
    model.save_pretrained("./qwen2vl-lora-adapter")
    processor.save_pretrained("./qwen2vl-lora-adapter")
    print("Adaptador LoRA salvo em ./qwen2vl-lora-adapter")

Writing train.py


In [11]:
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 train.py

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Resolving data files: 100%|██████████████████| 17/17 [00:00<00:00, 24604.27it/s]
model.safetensors.index.json: 100%|████████| 56.4k/56.4k [00:00<00:00, 99.7MB/s]
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 2 files:   0%|                                   | 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|        |  0.00B / 4.42GB            
Reconstructing (incomplete total...):  10%|▊       |  457MB / 4.42GB,  354MB/s  

Fetching 2 files:  50%|█████████████▌             | 1/2 [00:04<00:04,  4.24s/it]

Fetching 2 files:  50%|█████████████▌             | 1/2 [00:

## 8) Avaliação

Métrica: **ANLS** (*Average Normalized Levenshtein Similarity*), a métrica padrão do DocVQA. Para cada pergunta, compara-se a resposta do modelo com **cada resposta aceita**, usando a similaridade de Levenshtein normalizada; similaridades abaixo de 0,5 contam como zero, e vale a melhor. O resultado é a média sobre as perguntas (de 0 a 1).

Atenção ao interpretar:
- O número aqui **não é comparável** ao reportado oficialmente para o modelo: a resolução é limitada (`MAX_PIXELS`) e o conjunto de avaliação é pequeno.
- Parte do ganho do LoRA pode vir de **aprender o formato curto** das respostas, e não de "ler melhor". É por isso que existe o cenário **base + prompt**.
- Use um `N_EVAL` maior (200 a 500) para um número mais confiável.

In [12]:
from rapidfuzz.distance import Levenshtein

def anls(pred, golds, threshold=0.5):
    pred = str(pred).lower().strip()
    best = 0.0
    for g in golds:
        nl = Levenshtein.normalized_distance(pred, str(g).lower().strip())
        best = max(best, 1.0 - nl if nl < threshold else 0.0)
    return best


@torch.no_grad()
def generate_answer(model, example, suffix="", max_new_tokens=64):
    conv = build_conversation(example, with_answer=False, suffix=suffix)
    prompt = processor.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[example["image"].convert("RGB")], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    return text.strip()


def evaluate_model(model, data, label="", suffix=""):
    model.eval()
    rows, t0 = [], time.time()
    for ex in tqdm(data, desc=f"Avaliando {label}"):
        pred = generate_answer(model, ex, suffix=suffix)
        rows.append({
            "question": ex["question"],
            "gold": " | ".join(ex["answers"]),
            "pred": pred,
            "anls": anls(pred, ex["answers"]),
        })
    df = pd.DataFrame(rows)
    print(f"[{label}] ANLS = {df['anls'].mean():.3f} | palavras/resposta = {df['pred'].str.split().str.len().mean():.1f} "
          f"| tempo = {time.time() - t0:.0f}s")
    return df

In [13]:
from peft import PeftModel

base = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=DTYPE,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model = PeftModel.from_pretrained(base, "./qwen2vl-lora-adapter")
model.config.use_cache = True
torch.cuda.empty_cache()

# Cenário 1: modelo base (adaptador desligado), pergunta simples
with model.disable_adapter():
    df_base = evaluate_model(model, eval_dataset, label="base")

# Cenário 2: modelo base + prompt engineering
with model.disable_adapter():
    df_prompt = evaluate_model(model, eval_dataset, label="base + prompt", suffix=PROMPT_SUFFIX)

# Cenário 3: modelo com LoRA
df_lora = evaluate_model(model, eval_dataset, label="LoRA")

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at './qwen2vl-lora-adapter'

In [ ]:
# Inspeção qualitativa: lado a lado
comp = pd.DataFrame({
    "Pergunta": df_base["question"],
    "Esperado": df_base["gold"],
    "Base": df_base["pred"],
    "Base + prompt": df_prompt["pred"],
    "LoRA": df_lora["pred"],
    "ANLS LoRA": df_lora["anls"].round(2),
})
comp.head(15)

In [ ]:
# Só os erros do LoRA (ANLS < 0.5), para identificar o padrão:
# erro de leitura visual (o modelo "não viu" o texto) ou de formato da resposta?
erros = comp[comp["ANLS LoRA"] < 0.5]
print(f"{len(erros)} erro(s) de {len(comp)} exemplos")
erros

## 9) Relatório de custo (alimenta a discussão de viabilidade)

Guarde esta tabela para cada configuração testada (`USE_4BIT` ligado e desligado, `MAX_PIXELS` diferentes). Ela mostra o que o LoRA realmente custou.

A **estimativa de tempo** extrapola o tempo por exemplo do experimento para o treino completo.

In [ ]:
import json
cost = json.load(open("cost.json"))
relatorio = pd.DataFrame([cost]).T.rename(columns={0: "valor"})
print(relatorio)

n_full = 39463
seg_por_exemplo = cost["tempo_treino_s"] / (cost["exemplos_treino"] * cost["epocas"])
print(f"\nTempo por exemplo (por época): {seg_por_exemplo:.2f} s")
for ep in (1, 2, 3):
    horas = seg_por_exemplo * n_full * ep / 3600
    print(f"Estimativa para o treino completo com {ep} época(s): ~{horas:.1f} h")

## 10) Dicas para adaptar a outros casos

- **Memória (OOM):** reduza `MAX_PIXELS`, mantenha `BATCH_SIZE=1`, e confira o pico de memória no relatório. A resolução é, em geral, o fator que mais pesa em VLMs.
- **Rank do LoRA (`r`):** comece com 16; para tarefas mais difíceis, teste 32 ou 64 (mais capacidade, mais memória e risco de *overfitting*).
- **Outros VLMs:** a mesma lógica vale para Idefics, PaliGemma e LLaVA; mudam a classe do modelo, o *processor* e os `target_modules`.
- **Reprodutibilidade:** registre as versões das bibliotecas (célula 3), a GPU, a semente e a configuração usada.